In [1]:
# SYSTEM + PATH CONFIGURATION

import os

# Must be set BEFORE importing torch or matplotlib, to avoid the OMP Error #15
# kernel crash (duplicate OpenMP runtime — see test_dataloader.ipynb notes)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import torch
import importlib

# PROJECT_ROOT = r'C:\Users\aliyu\Desktop\architectural-inductive-bias-study'

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("✓ System Configuration:")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Core modules found: {os.listdir(os.path.join(PROJECT_ROOT, 'core_modules'))}")

✓ System Configuration:
Python version: 3.10.20
PyTorch version: 2.5.1+cu121
CUDA available: True
Project root: C:\Users\aliyu\Desktop\autodl-fs
Core modules found: ['cnn.py', 'dataloader.py', 'gradcam.py', 'mlp.py', 'model.py', 'rnn.py', '__init__.py', '__pycache__']


In [2]:
# MODEL BUILD + STRUCTURE CHECK

from core_modules.model import build_model
import importlib

CONFIG_MODULE = 'configs.flower_rnn'
config_module = importlib.import_module(CONFIG_MODULE)
CONFIG = config_module.CONFIG

# build_model() now only takes device + config —
# num_classes comes from CONFIG internally, fine_tune was removed
# since it was specific to pretrained EfficientNet backbones
model = build_model(
    device=CONFIG['device'],
    config=CONFIG
)

print("\n⚙️ Model Configuration:")
for k, v in CONFIG.items():
    print(f"{k:>20}: {v}")

Model: RNN
Model Configuration:
├── Trainable params: 412,421
├── Frozen params: 0
└── Output head: Sequential(
  (0): Linear(in_features=128, out_features=128, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=5, bias=True)
)


⚙️ Model Configuration:
             dataset: flowers
          model_name: rnn
         num_classes: 5
          batch_size: 64
              epochs: 40
                  lr: 0.001
          input_size: 128
              device: cuda
        project_root: C:\Users\aliyu\Desktop\autodl-fs
           train_dir: C:\Users\aliyu\Desktop\autodl-fs\datasets/flowers/train
             val_dir: C:\Users\aliyu\Desktop\autodl-fs\datasets/flowers/val
            test_dir: C:\Users\aliyu\Desktop\autodl-fs\datasets/flowers/test
 phase1_augmentation: none
        weight_decay: 0.0005
        dropout_rate: 0.3
          adam_beta1: 0.9
          adam_beta2: 0.999
            adam_eps: 1e-08
     label_smoothi

In [4]:
# Build model and verify structure

try:
    model = build_model(
        device=CONFIG['device'],
        config=CONFIG
    )
    print("\n✓ Model Validation Passed:")
    print(f"- Device: {next(model.parameters()).device}")
    print(f"- Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"- Frozen params: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")
    print(f"- Total params: {sum(p.numel() for p in model.parameters()):,}")

    # Print classifier output size — use [-1] (last layer) rather than [1],
    # since CNN/MLP/RNN's classifier structure (Flatten, Linear, ReLU, Dropout, Linear)
    # differs from EfficientNet's (Dropout, Linear), so index 1 no longer
    # reliably points at the final output layer
    if hasattr(model, 'classifier'):
        print(f"- Classifier output: {model.classifier[-1].out_features} units")
    elif hasattr(model, 'fc'):
        print(f"- Classifier output: {model.fc.out_features} units")
    else:
        print("- Classifier output: Unknown structure")

except Exception as e:
    print(f"\n✗ Validation Failed: {e}")

Model: RNN
Model Configuration:
├── Trainable params: 412,421
├── Frozen params: 0
└── Output head: Sequential(
  (0): Linear(in_features=128, out_features=128, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=5, bias=True)
)


✓ Model Validation Passed:
- Device: cuda:0
- Trainable params: 412,421
- Frozen params: 0
- Total params: 412,421
- Classifier output: 5 units
